# 05 – Tren en lineær klassifikator

**Tid:** 60 minutter. **Mål:** samle bildedata, modell, tap og optimizer i en
synlig treningsløkke. Nå skal modellen faktisk lære å gjenkjenne sifre.

Denne notatboken har sin egen kjerne. Du skal gjenbruke modellideen fra
notatbok 03, ikke variablene fra den gamle kjernen.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn


In [ ]:
from pytorch_101.utils import download_mnist

treningsdata, testdata = download_mnist()
print(f"Treningsbilder: {len(treningsdata)}, testbilder: {len(testdata)}")


## Eksempel: en komplett treningsløkke på et lite problem

Her lærer en lineær modell sammenhengen `y = 2*x + 1` fra noen få punkter.
Vi bruker kvadratisk tap fordi dette er et tallproblem, ikke klassifikasjon.
Følg de fem stegene: nullstill, prediker, beregn tap, beregn gradienter, oppdater.


In [ ]:
x = torch.tensor([[-1.0], [0.0], [1.0], [2.0]])
y = 2 * x + 1
liten_modell = nn.Linear(1, 1)
liten_optimizer = torch.optim.SGD(liten_modell.parameters(), lr=0.1)
kvadratisk_tap = nn.MSELoss()
for steg in range(60):
    liten_optimizer.zero_grad()
    prediksjon = liten_modell(x)
    tap = kvadratisk_tap(prediksjon, y)
    tap.backward()
    liten_optimizer.step()
print("Lært vekt:", liten_modell.weight.item())
print("Lært konstantledd:", liten_modell.bias.item())


## Grupper til trening og evaluering

Som i notatbok 03 oppretter vi datalasterne direkte. Treningsbildene
stokkes. Testbildene brukes bare til evaluering og beholder rekkefølgen.
Ingen global frøsetting er nødvendig; målingene kan variere litt.


In [ ]:
from torch.utils.data import DataLoader

treningslaster = DataLoader(treningsdata, batch_size=64, shuffle=True)
testlaster = DataLoader(testdata, batch_size=64, shuffle=False)


## Oppgave 1: Sett opp MNIST-modellen

Lag `modell` med `nn.Flatten()` og `nn.Linear(784, 10)`, som i notatbok 03.
Lag `tapsfunksjon = nn.CrossEntropyLoss()` og `optimizer` med
`torch.optim.SGD`, modellens parametere og læringsrate 0.1.

<details>
<summary>Hint</summary>

Optimizer må få `modell.parameters()` fra akkurat den modellen du trener.

</details>


In [ ]:
modell = nn.Sequential(nn.Flatten(), nn.Linear(784, 10))
tapsfunksjon = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(modell.parameters(), lr=0.1)
print(modell)


**Kontroll:** Kjør denne cellen når du har løst oppgaven. En feilmelding betyr at du må undersøke løsningen.


In [ ]:
assert modell(torch.zeros(2, 1, 28, 28)).shape == (2, 10)


## Fra ett steg til flere epoker

Én **epoke** er én gjennomgang av alle treningsbildene. Datalasteren gir én
gruppe om gangen. Den siste gruppen kan være mindre, så bruk `len(bilder)`
i beregningene. `CrossEntropyLoss` gir gjennomsnittstap for gruppen.
For et riktig epokegjennomsnitt vekter vi derfor gruppetapet med gruppestørrelsen.

`modell.train()` setter modellen i treningsmodus. Det er en god vane, selv om
akkurat vår lineære modell oppfører seg likt i trenings- og evalueringsmodus.


## Oppgave 2: Skriv treningsløkken

Tren modellen i fem epoker. Opprett `treningstap = []`.
For hver epoke:

1. Sett modellen i treningsmodus og start en tapsum på 0.
2. Gå gjennom `treningslaster` med `for bilder, etiketter in treningslaster`.
3. Nullstill gradientene, beregn skårer og tap, kall `backward()` og `step()`.
4. Legg `tap.item() * len(bilder)` til tapsummen.
5. Del tapsummen på antall treningsbilder og lagre resultatet i `treningstap`.

Skriv ut gjennomsnittstapet etter hver epoke. Kjør modellcellen over på nytt
hvis du vil starte treningen fra begynnelsen.

<details>
<summary>Hint</summary>

Bruk treningsmønsteret fra det lille eksemplet. Bare modell, data og tapsfunksjon er annerledes.

</details>


In [ ]:
treningstap = []
for epoke in range(5):
    modell.train()
    tapsum = 0.0
    for bilder, etiketter in treningslaster:
        optimizer.zero_grad()
        skaarer = modell(bilder)
        tap = tapsfunksjon(skaarer, etiketter)
        tap.backward()
        optimizer.step()
        tapsum += tap.item() * len(bilder)
    gjennomsnittstap = tapsum / len(treningsdata)
    treningstap.append(gjennomsnittstap)
    print(f"Epoke {epoke + 1}: treningstap {gjennomsnittstap:.4f}")


**Kontroll:** Kjør denne cellen når du har løst oppgaven. En feilmelding betyr at du må undersøke løsningen.


In [ ]:
assert len(treningstap) == 5
assert all(np.isfinite(treningstap))
assert treningstap[-1] < treningstap[0]


### Sjekk forståelsen

Oppgi første og siste treningstap. Hva antyder utviklingen?
Hva er forskjellen på én gruppe, én parameteroppdatering og én epoke?
Hvorfor ganger vi gruppetapet med antall bilder før vi summerer?

Skriv svaret ditt i Markdown-cellen under. Bruk egne resultater der det er relevant.


**Mitt svar:**

Et eksempel er at tapet faller fra omtrent 1,01 til 0,41. Tallene
varierer med tilfeldig initialisering og stokking; oppgi dine egne.
Lavere tap betyr bedre tilpasning til treningsdata etter tapsfunksjonens
mål, men beviser ikke alene at modellen generaliserer.
Én gruppe inneholder flere bilder og gir én oppdatering i denne løkken.
Én epoke omfatter alle gruppene. Vi vekter etter antall bilder, slik
at en liten siste gruppe ikke teller like mye som en full gruppe.


## Eksempel: nøyaktighet fra skårer

`argmax(dim=1)` velger klasse per eksempel. Sammenligningen med etikettene
gir boolske verdier, som kan telles med `sum()`.
Ved evaluering bruker vi `modell.eval()` og `torch.no_grad()`:
førstnevnte velger evalueringsmodus, sistnevnte slår av gradientregistrering.
Vi verken beregner gradienter eller oppdaterer parametere på testdata.


In [ ]:
eksempelskaarer = torch.tensor([[3., 0., 1.], [0., 1., 4.], [0., 2., 1.]])
eksempeletiketter = torch.tensor([0, 1, 1])
gjett = eksempelskaarer.argmax(dim=1)
print("Prediksjoner:", gjett)
print("Nøyaktighet:", (gjett == eksempeletiketter).float().mean().item())


## Oppgave 3: Evaluer på testbilder

Sett modellen i evalueringsmodus. Under `torch.no_grad()`: gå gjennom
`testlaster`, beregn skårer, velg prediksjoner og tell riktige.
Lag `testnoyaktighet` som antall riktige delt på antall testbilder.
Skriv ut resultatet som prosent. Vis også `treningstap` som en kurve.

<details>
<summary>Hint</summary>

Start med `antall_riktige = 0`. Bruk `(prediksjoner == etiketter).sum().item()`.

</details>


In [ ]:
modell.eval()
antall_riktige = 0
with torch.no_grad():
    for bilder, etiketter in testlaster:
        skaarer = modell(bilder)
        prediksjoner = skaarer.argmax(dim=1)
        antall_riktige += (prediksjoner == etiketter).sum().item()
testnoyaktighet = antall_riktige / len(testdata)
print(f"Testnøyaktighet: {testnoyaktighet:.1%}")
figur, akse = plt.subplots()
akse.plot(range(1, 6), treningstap, marker="o")
akse.set(xlabel="Epoke", ylabel="Gjennomsnittlig treningstap", title="Den lineære modellen lærer")
plt.show()


**Kontroll:** Kjør denne cellen når du har løst oppgaven. En feilmelding betyr at du må undersøke løsningen.


In [ ]:
assert 0 <= testnoyaktighet <= 1


### Sjekk forståelsen

Hva ble testnøyaktigheten, og hva betyr den i antall riktige av 1 000 bilder?
Hva ville vært galt med å kalle `optimizer.step()` i testløkken?
Er `eval()` og `no_grad()` det samme? Forklar forskjellen.

Skriv svaret ditt i Markdown-cellen under. Bruk egne resultater der det er relevant.


**Mitt svar:**

Et eksempel er 91,4 %, altså 914 riktige av 1 000 bilder. Det er ikke
en fasit: uten fast frø varierer modellen og resultatet. Bruk din egen
målte prosent og antall riktige. Oppdateringer på testdata ville gjort
testen del av treningen. `eval()` velger modellmodus, mens `no_grad()`
hindrer gradientregistrering. Ingen av dem oppdaterer parametere.
